# 01 · Muestra de audio FLEURS (`es_419`) por streaming

**Objetivo:** obtener una pequeña muestra de audio en español (configuración `es_419`) desde [`google/fleurs`](https://huggingface.co/datasets/google/fleurs) **sin descargar el corpus completo**, guardarla como WAV y generar un manifest reproducible.

**Nota:** FLEURS no expone `speaker_id`; la columna queda vacía (`None`) y así se documenta. No se simula un identificador que no existe.

## 1. Entorno y dependencias

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT_OVERRIDE: Path | None = None

def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _is_project_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'requirements' / 'dataset.txt').is_file()

def _find_drive_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        return PROJECT_ROOT_OVERRIDE.expanduser().resolve()
    matches = sorted({src_dir.parent for src_dir in Path('/content/drive').glob('**/src')
                      if _is_project_root(src_dir.parent)})
    if len(matches) == 1:
        return matches[0]
    if matches:
        found = '\n - '.join(str(path) for path in matches)
        raise SystemExit(f'Se encontraron varios proyectos:\n - {found}\nAsigna uno a PROJECT_ROOT_OVERRIDE.')
    raise SystemExit('No se encontro IA-Proyecto. Monta la cuenta correcta de Drive o asigna PROJECT_ROOT_OVERRIDE.')

if _is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = _find_drive_project_root()
else:
    PROJECT_ROOT = Path.cwd()

if not _is_project_root(PROJECT_ROOT):
    raise SystemExit(f'No se encontro la raiz del proyecto en {PROJECT_ROOT}.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
%pip install --quiet -r requirements/dataset.txt

### 1.1 Imports

In [ ]:
import pandas as pd
from IPython.display import Audio, display

from src.dataset import fleurs
from src.dataset import manifest as m

## 2. Cargar FLEURS en streaming

Con `streaming=True` no se descarga el corpus completo (que en es_419 pesa varios GB): los ejemplos se descargan bajo demanda mientras se itera.

In [ ]:
stream = fleurs.stream_fleurs()
print("Tipo de dataset:", type(stream).__name__)

## 3. Tomar 20 muestras reproducibles

El barajado con ventana garantiza variación y, con la misma semilla, los mismos ejemplos.

In [ ]:
N_SAMPLES = 20
SEED = 42
samples = fleurs.take_samples(stream, n=N_SAMPLES, seed=SEED)
print(f"Muestras tomadas: {len(samples)}")
print("Claves de una muestra:", sorted(samples[0].keys()))

## 4. Explorar cada muestra

Cada muestra trae `audio` (array + tasa de muestreo), transcripción (`raw_transcription` vs `transcription` normalizada), idioma, género y duración calculada localmente.

In [ ]:
rows = []
for sample in samples:
    audio = sample["audio"]
    rows.append({
        "id": sample["id"],
        "transcription_es": fleurs.get_transcription(sample),
        "raw_transcription": sample.get("raw_transcription", ""),
        "normalized": sample.get("transcription", ""),
        "gender": sample.get("gender", None),
        "language": sample.get("language", None),
        "sample_rate": audio["sampling_rate"],
        "duration_sec": round(fleurs.audio_duration_sec(audio), 3),
    })
explore = pd.DataFrame(rows)
display(explore)

## 5. Reproducir audios dentro del notebook

Se reproducen 3 muestras (señal ya decodificada, no es necesario archivo).

In [ ]:
AUDIOS_TO_PLAY = 3
for sample in samples[:AUDIOS_TO_PLAY]:
    audio = sample["audio"]
    display(Audio(data=audio["array"], rate=audio["sampling_rate"], autoplay=False))
    print("Transcripción:", fleurs.get_transcription(sample))

## 6. Guardar muestras en WAV y construir el manifest

- Solo se guardan las 20 muestras seleccionadas (no el corpus completo) en `data/raw/audio/`.
- Se escriben como **WAV mono a 16 kHz**, calidad suficiente para ASR y como referencia de voice cloning.
- Las rutas del manifest son relativas al proyecto.

In [ ]:
manifest = fleurs.build_fleurs_manifest(samples, audio_dir="data/raw/audio")
display(manifest)

In [ ]:
manifest_path = m.save_manifest(manifest, m.FLEURS_MANIFEST_PATH)

In [ ]:
import soundfile as sf

first_audio = manifest.iloc[0]["audio_path"]
data, sr = sf.read(first_audio)
print(f"Archivo: {first_audio}")
print(f"tasa={sr} Hz | muestras={data.shape[0]} | duración={data.shape[0]/sr:.3f} s")
display(Audio(data=data, rate=sr))

## 7. Resumen

In [ ]:
from pathlib import Path

wav_files = list(Path("data/raw/audio").glob("fleurs_es_419_*.wav"))
summary = pd.DataFrame({
    "metrica": ["muestras", "duración total (s)", "sample rate (Hz)", "formato", "speaker_id", "archivos wav"],
    "valor": [
        len(manifest),
        round(float(manifest["duration_sec"].sum()), 2),
        16000,
        "WAV (float32, mono)",
        "No disponible en FLEURS (None)",
        len(wav_files),
    ],
})
display(summary)

**Nota de diseño:** FLEURS no proporciona un `speaker_id`. La columna queda a `None` y no se inventa; si en el futuro se usa otro dataset con hablante, el manifest ya contempla la columna.